# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Imhs14/flyrank-ml-internship-starter-main/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

For each finding, ask constructively:
1. Where does the label come from — an observed outcome, or a rule someone already wrote?
2. Does the validation design actually test the claim being made, or something weaker?
3. Would this number survive being recomputed on a grouped or time-aware split?

Frame it as "how to make this stronger," never as a gotcha — you're practicing the rigor you'd
want applied to your own work, not scoring points.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

I re-ran my Week-5 clustering approach under both a naive random split and my grouped-by-client
split. The gap here is small (0.565 vs 0.590 — grouped is actually slightly HIGHER, not lower).
I expected the random split to look artificially inflated, since same-client pages could leak
across train/test — that's not what happened here. My best guess why: my features (CTR, position,
word count, engagement) describe individual page behavior, not client identity, so a client's
pages don't cluster together strongly enough for random splitting to create much advantage. A
small or reversed gap is still a valid finding — not every honest-split audit uncovers dramatic
inflation, and reporting that plainly is more trustworthy than manufacturing a bigger story.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os
while not os.path.isdir("data/raw") and os.getcwd() != "/":
    os.chdir("..")
assert os.path.exists("data/raw/content_refresh_anonymized.csv")

import pandas as pd, numpy as np
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

df = pd.read_csv('data/raw/content_refresh_anonymized.csv')
for col in ['word_count', 'scroll_rate']:
    df[f'{col}_was_missing'] = df[col].isna().astype(int)
    df[col] = df[col].fillna(df[col].median())
df['is_declining'] = (df['trend_direction'] == 'down').astype(int)

feat_cols = ['avg_position', 'ctr', 'impressions_90d', 'word_count', 'content_age_days', 'engagement_rate', 'scroll_rate']
K = 200

def evaluate_split(train, test, label):
    train, test = train.copy(), test.copy()
    scaler = StandardScaler().fit(train[feat_cols])
    Xtr, Xte = scaler.transform(train[feat_cols]), scaler.transform(test[feat_cols])
    km = KMeans(n_clusters=5, random_state=42, n_init=10).fit(Xtr)
    train['cluster'], test['cluster'] = km.labels_, km.predict(Xte)
    risky = train.groupby('cluster')['is_declining'].mean().idxmax()
    centroid = km.cluster_centers_[risky]
    test_c = test[test['cluster'] == risky].copy()
    test_c['dist'] = np.linalg.norm(scaler.transform(test_c[feat_cols]) - centroid, axis=1)
    prec = test_c.sort_values('dist').head(K)['is_declining'].mean()
    print(f"{label}: precision@{K} = {prec:.3f}  (n in risky cluster = {len(test_c)})")
    return prec

# BEFORE — naive random row split, same client can leak across train/test
train_r, test_r = train_test_split(df, test_size=0.25, random_state=42)
prec_before = evaluate_split(train_r, test_r, "BEFORE (random split)")

# AFTER — grouped by client, same as Week 5
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
tr_idx, te_idx = next(gss.split(df, groups=df['client_id']))
prec_after = evaluate_split(df.iloc[tr_idx], df.iloc[te_idx], "AFTER (grouped by client)")

print(f"\nGap: {prec_before - prec_after:+.3f}")

BEFORE (random split): precision@200 = 0.565  (n in risky cluster = 3645)
AFTER (grouped by client): precision@200 = 0.590  (n in risky cluster = 2649)

Gap: -0.025


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# checklist: confirm none of the real feature columns are label-derived, future-window, or product flags
excluded_cols = ["trend_direction", "trend_pct", "is_declining_label", "is_declining",
                  "needs_ctr_fix", "health_score", "priority_score", "action_type"]
print("Final feature columns:", feat_cols)
print("Any excluded columns present in features?", [c for c in feat_cols if c in excluded_cols])

# deliberate collapse test — add the label-derived column back in on purpose
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
tr_idx, te_idx = next(gss.split(df, groups=df['client_id']))
train, test = df.iloc[tr_idx].copy(), df.iloc[te_idx].copy()

honest = evaluate_split(train, test, "Honest (feat_cols only)")

feat_cols_leaked = feat_cols + ['is_declining']
def evaluate_leak(train, test):
    train, test = train.copy(), test.copy()
    scaler = StandardScaler().fit(train[feat_cols_leaked])
    Xtr, Xte = scaler.transform(train[feat_cols_leaked]), scaler.transform(test[feat_cols_leaked])
    km = KMeans(n_clusters=5, random_state=42, n_init=10).fit(Xtr)
    train['cluster'], test['cluster'] = km.labels_, km.predict(Xte)
    risky = train.groupby('cluster')['is_declining'].mean().idxmax()
    centroid = km.cluster_centers_[risky]
    test_c = test[test['cluster'] == risky].copy()
    test_c['dist'] = np.linalg.norm(scaler.transform(test_c[feat_cols_leaked]) - centroid, axis=1)
    prec = test_c.sort_values('dist').head(K)['is_declining'].mean()
    print(f"WITH is_declining leaked in: precision@{K} = {prec:.3f}")
    return prec

leaked = evaluate_leak(train, test)
print(f"\nJump: {leaked - honest:+.3f}")

Final feature columns: ['avg_position', 'ctr', 'impressions_90d', 'word_count', 'content_age_days', 'engagement_rate', 'scroll_rate']
Any excluded columns present in features? []
Honest (feat_cols only): precision@200 = 0.590  (n in risky cluster = 2649)
WITH is_declining leaked in: precision@200 = 1.000

Jump: +0.410


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

**My boldest sentence from Week 5:** "Cluster 1 identifies at-risk content that's likely to
decline."

**Rewritten with safe language:** "In this dataset, Cluster 1 — pages with moderate traffic,
weak CTR, and low engagement — showed an observed decline rate of 63% in the training window.
On held-out clients, ranking within this cluster by closeness to its centroid reached a measured
precision@200 of 59%, above the 52% base rate but below the Week-4 rule's 73%. This is
directional, decision-support evidence that this page profile is worth reviewing first — not a
guarantee that any individual page in this cluster will decline, and not a causal claim about why."

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.